<a href="https://colab.research.google.com/github/yadavvipin02108-lang/Hospital_Agent/blob/main/multi_pdf_rag_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install pypdf
!pip -q install sentence-transformers
!pip -q install faiss-cpu
!pip -q install transformers
!pip -q install accelerate
!pip -q install langchain-community
!pip -q install langchain-text-splitters


In [ ]:
from google.colab import files
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import pipeline

# =========================
# Upload PDFs
# =========================

print("Uploads PDF files")
uploaded = files.upload()

pdf_files = [f for f in uploaded.keys() if f.lower().endswith('.pdf')]

# =========================
# Read PDFs
# =========================

documents = []

for pdf in pdf_files:
    reader = PdfReader(pdf)

    for page_no, page in enumerate(reader.pages):

        text = page.extract_text()

        if text:
            documents.append({
                "source": pdf,
                "page": page_no + 1,
                "text": text
            })

print("Pages Loaded:", len(documents))

# =========================
# Split Text
# =========================

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = []
metadata = []

for doc in documents:

    split_chunks = splitter.split_text(doc["text"])

    for chunk in split_chunks:
        chunks.append(chunk)

        metadata.append({
            "source": doc["source"],
            "page": doc["page"]
        })

print("Chunks Created:", len(chunks))

# =========================
# Embedding Model
# =========================

embed_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

embeddings = embed_model.encode(
    chunks,
    convert_to_numpy=True
)

print("Embeddings Generated")

# =========================
# FAISS Index
# =========================

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("FAISS Index Ready")

# =========================
# Load LLM
# =========================
generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    max_length=512
)

print("LLM Loaded")

# =========================
# RAG Function
# =========================

def ask_pdf(question, top_k=3):

    query_embedding = embed_model.encode(
        [question],
        convert_to_numpy=True
    )

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    context = ""

    sources = []

    for idx in indices[0]:

        context += chunks[idx] + "\n\n"

        sources.append(
            f"{metadata[idx]['source']} | Page {metadata[idx]['page']}"
        )

    prompt = f"""
Use the following context to answer.

Context:
{context}

Question:
{question}

Answer:
"""

    response = generator(prompt)

    answer = response[0]["generated_text"]

    print("\n====================")
    print("ANSWER:")
    print(answer)

    print("\nSOURCES:")

    for src in list(set(sources)):
        print("-", src)

    print("====================\n")

# =========================
# Chat Loop
# =========================

print("\nMulti PDF RAG Chatbot Ready")
print("Type exit to stop\n")

while True:

    question = input("You: ")

    if question.lower() == "exit":
        print("Chat Ended")
        break

    ask_pdf(question)

Uploads PDF files


Saving Krishna_Pal_Resume (1).pdf to Krishna_Pal_Resume (1) (1).pdf
Pages Loaded: 1
Chunks Created: 1


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings Generated
FAISS Index Ready

Multi PDF RAG Chatbot Ready
Type exit to stop


ANSWER:

Use the following context to answer.

Context:
GET IN TOUCH!
Mobile:
 
+91-8532980561
Email:
 
yadavvipin02108@gmail.com
SKILLS
-
Coding Standards
-
Python
-
Javascript
-
Javascript Programmer
-
C Programming Language
Krishna Pal
PERSONAL DETAILS
Date of Birth
May 4, 2006
Male
EDUCATION
Graduation
Course
B.Tech / B.E. ( Computer Science and Engineering (CSE) )
College
Rakshpal Bahadur College of Engineering and Technology, Bareilly
Schooling
 
Class 
XII
 
Class 
X
Board Name
Uttar Pradesh
Uttar Pradesh
Medium
Hindi
Hindi
Year of Passing
2023
2021
Score
70%
85%
ACHIEVEMENTS
-
All rounder in B.Tech / B.E.
-
School topper in school
Current Location
Aonla

GET IN TOUCH!
Mobile:
 
+91-8532980561
Email:
 
yadavvipin02108@gmail.com
SKILLS
-
Coding Standards
-
Python
-
Javascript
-
Javascript Programmer
-
C Programming Language
Krishna Pal
PERSONAL DETAILS
Date of Birth
May 4, 2006
Male
EDUCATION
